<a href="https://colab.research.google.com/github/hadi-hosseini/bandit/blob/main/Bandit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [361]:
import math
import cvxpy as cp
import numpy as np
from scipy.stats import ortho_group
from tqdm import tqdm

# Parameters
k = 4
d = 1000
n_samples = 1000
sigma = 1.0
T = 100

orthogonal_matrix = ortho_group.rvs(dim=d)
mu = orthogonal_matrix[:k]
print(mu)

[[-0.05378758  0.062747   -0.05293557 ... -0.00791095  0.00696041
   0.03106025]
 [ 0.01855808 -0.00349503  0.03306984 ...  0.01092765 -0.03523285
  -0.05421913]
 [ 0.07326427  0.04236293  0.03695527 ... -0.03960933 -0.00167786
  -0.03504239]
 [-0.00498633  0.0002945   0.02962197 ...  0.01971227 -0.00588511
  -0.02749781]]


In [362]:
# Verify orthogonality
for i in range(k):
    for j in range(i+1, k):
        print(f"Dot product μ_{i+1}·μ_{j+1}: {np.dot(mu[i], mu[j])}")

Dot product μ_1·μ_2: -2.6454533008646308e-17
Dot product μ_1·μ_3: -3.7730235602495554e-17
Dot product μ_1·μ_4: -3.903127820947816e-18
Dot product μ_2·μ_3: -3.144186300207963e-17
Dot product μ_2·μ_4: 2.42861286636753e-17
Dot product μ_3·μ_4: -6.396792817664476e-18


In [363]:
# create offline dataset
def create_logged_data(k, d, n_samples, sigma, mu):
  samples = []
  for i in range(k):
    # Generate samples from N(μ_i, σ²I)
    samples.append(np.random.normal(loc=mu[i], scale=sigma, size=(n_samples, d)))
  return samples

logged_data = create_logged_data(k, d, n_samples, sigma, mu)

In [322]:
# implement UCB algorithm
class UCBAlgorithm:
    def __init__(self, k, d, true_means, logged_data, perturbation):
        self.k = k
        self.d = d
        self.true_means = true_means
        self.logged_data = logged_data

        self.N = np.zeros(k)
        self.total_rewards = np.zeros(k)
        self.empirical_rewards = np.zeros(k)
        self.empirical_means = np.zeros((k, d))
        self.perturbation = perturbation

    def get_reward(self, x):
        return np.dot(self.true_means[0] + self.perturbation, x)

    def select_arm(self, t):
        if t < self.k:
            return t

        ucb_values = np.zeros(self.k)
        for j in range(self.k):
            mean_term = self.empirical_rewards[j]
            confidence_bound = math.sqrt((2 * math.log(t)) / self.N[j])

            ucb_values[j] = mean_term + confidence_bound

        return np.argmax(ucb_values)

    def update(self, arm, reward):
        self.N[arm] += 1
        self.total_rewards[arm] += reward
        self.empirical_rewards[arm] = self.total_rewards[arm] / self.N[arm]

    def run(self, T):
        rewards = np.zeros(T)
        chosen_arms = np.zeros(T, dtype=int)

        for t in tqdm(range(T)):
            arm = self.select_arm(t)
            sample = self.logged_data[arm][int(self.N[arm])]
            reward = self.get_reward(sample)

            self.update(arm, reward)
            self.empirical_means[arm] = self.empirical_means[arm] + (sample - self.empirical_means[arm])/self.N[arm]

            rewards[t] = reward
            chosen_arms[t] = arm

        return rewards, chosen_arms

In [323]:
# run UCB without perturbation
perturbation = 0.0
ucb = UCBAlgorithm(k, d, mu, logged_data, perturbation)
rewards, chosen_arms = ucb.run(T)
print("\nNumber of pulls per arm:", ucb.N)
print(chosen_arms)

100%|██████████| 100/100 [00:00<00:00, 38272.69it/s]


Number of pulls per arm: [78. 13.  9.]
[0 1 2 2 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 2 0 0 0 0 0 0 0 0 0 1 1 0
 0 2 0 1 0 2 2 0 0 0 0 1 0 0 0 0 2 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 1 0 0 0 0 0 1 2 2 0 0 0 0 0 0 0 0 0 0 0 0]


In [324]:
def random_perturbation(d, epsilon):
    perturbation = np.random.randn(d)
    perturbation = epsilon * perturbation / np.linalg.norm(perturbation)
    return perturbation

epsilon = 0.5
random_perturbation = random_perturbation(d, epsilon)
ucb = UCBAlgorithm(k, d, mu, logged_data, random_perturbation)
rewards, chosen_arms = ucb.run(T)
print("\nNumber of pulls per arm:", ucb.N)
print(chosen_arms)

100%|██████████| 100/100 [00:00<00:00, 25893.96it/s]


Number of pulls per arm: [84. 10.  6.]
[0 1 2 2 1 0 0 0 0 0 1 1 0 0 0 2 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 1
 0 0 0 0 1 1 0 0 2 2 0 0 0 0 0 0 0 0 0 0 2 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [414]:
# find adversary perturbation
# should learn this perturbation based on the logged data

class FindPerturbation:
    def __init__(self, k, d, true_means, logged_data, epsilon, qp=False, M=1):
        self.k = k
        self.d = d
        self.true_means = true_means
        self.logged_data = logged_data
        self.epsilon = epsilon
        self.qp = qp
        self.M = M # alternatives

        self.N = np.zeros(k)
        self.empirical_means = np.zeros((k, d))
        self.perturbation = None
        self.history = []
        self.all_perturbs = []
        self.turn = 1

    def select_arm(self, t):
        if t < self.k:
            return t

        ### targetted
        # turn = self.turn

        ### untargetted
        self.turn += 1
        if self.turn == k:
          self.turn = 1

        return self.turn

    def find_perturbation_with_l2_ball_optimal_only(self, arm, t):
        x = cp.Variable(self.d)

        d_0 = self.empirical_means[arm] - self.empirical_means[0]
        c_0 = (math.sqrt(2 * math.log(t) / self.N[0]) - math.sqrt(2 * math.log(t) / self.N[arm])) - np.dot(self.true_means[0], d_0)
        self.history.append((d_0, c_0))


        constraints = []
        for (d_0, c_0) in self.history:
            constraints.append(x @ d_0 >= c_0 + 1e-6)
        constraints.append(cp.norm(x, 2) <= self.epsilon)
        prob = cp.Problem(cp.Minimize(0), constraints)
        prob.solve()

        if prob.status == 'optimal':
          self.all_perturbs.append(x.value)
          return x.value
        else:
          return None

    def find_perturbation_with_l2_ball(self, arm, t):
        x = cp.Variable(self.d)

        for j in range(self.k):
            if j != arm:
              d_j = self.empirical_means[arm] - self.empirical_means[j]
              c_j = (math.sqrt((2 * math.log(t)) / self.N[j]) - math.sqrt((2 * math.log(t)) / self.N[arm])) - np.dot(self.true_means[0], d_j)
              self.history.append((d_j, c_j))

        constraints = []
        for (d_j, c_j) in self.history:
            constraints.append(x @ d_j >= c_j + 1e-6)
        if qp:
          objective = cp.Minimize(cp.norm(x, 2))
          prob = cp.Problem(objective, constraints)
        else:
          constraints.append(cp.norm(x, 2) <= self.epsilon)
          prob = cp.Problem(cp.Minimize(0), constraints)
        prob.solve()

        if prob.status == 'optimal':
          self.all_perturbs.append(x.value)
          return x.value
        else:
          return None


    def run(self, T, mode=1):
        chosen_arms = np.zeros(T, dtype=int)

        for t in tqdm(range(T)):
            arm = self.select_arm(t)

            if t >= self.k and ((t-k) % self.M == 0):
              if mode == 1: # check all inequalities
                perturbation = self.find_perturbation_with_l2_ball(arm, t)
              elif mode == 2: # check just optimal inequalities
                perturbation = self.find_perturbation_with_l2_ball_optimal_only(arm, t)


              if perturbation is None:
                return chosen_arms

              self.perturbation = perturbation

            sample = self.logged_data[arm][int(self.N[arm])]
            self.N[arm] += 1
            self.empirical_means[arm] = self.empirical_means[arm] + (sample - self.empirical_means[arm])/self.N[arm]

            chosen_arms[t] = arm

        return chosen_arms

## ABLATION 1 (CHECK ALL INEQUALITIES)

In [415]:
# check all inequalities (mode=1)
epsilon = 0.5
qp = False
find_perturbation = FindPerturbation(k, d, mu, logged_data, epsilon, qp)
chosen_arms = find_perturbation.run(T, mode=1)
print("\nNumber of pulls per arm:", find_perturbation.N)
print(chosen_arms)

100%|██████████| 100/100 [01:25<00:00,  1.18it/s]


Number of pulls per arm: [ 1. 33. 33. 33.]
[0 1 2 3 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1
 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2
 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1]


In [416]:
perturbation = find_perturbation.perturbation
norm = np.linalg.norm(perturbation)
print(norm)

0.47432609124222314


In [417]:
# all_perturbs = find_perturbation.all_perturbs

# for hist in all_perturbs:
#   ucb_with_perturb = UCBAlgorithm(k, d, mu, logged_data, hist)
#   rewards, chosen_arms = ucb_with_perturb.run(T)
#   print("\nNumber of pulls per arm:", ucb_with_perturb.N)
#   print(chosen_arms)

In [418]:
# run UCB with perturbation
ucb_with_perturb = UCBAlgorithm(k, d, mu, logged_data, perturbation)
rewards, chosen_arms = ucb_with_perturb.run(T)
print("\nNumber of pulls per arm:", ucb_with_perturb.N)
print(chosen_arms)

100%|██████████| 100/100 [00:00<00:00, 29330.80it/s]


Number of pulls per arm: [ 1. 33. 33. 33.]
[0 1 2 3 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1
 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2
 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1]


In [419]:
print(len(find_perturbation.history))

288


## ABLATION 2 (CHECK ONLY OPTIMAL INEQUALITIES)

In [420]:
# check just optimal inequalities (mode=2)
epsilon = 0.5
qp = False
find_perturbation = FindPerturbation(k, d, mu, logged_data, epsilon, qp)
chosen_arms = find_perturbation.run(T, mode=2)
print("\nNumber of pulls per arm:", find_perturbation.N)
print(chosen_arms)

100%|██████████| 100/100 [00:22<00:00,  4.52it/s]


Number of pulls per arm: [ 1. 33. 33. 33.]
[0 1 2 3 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1
 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2
 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1]


In [421]:
perturbation = find_perturbation.perturbation
norm = np.linalg.norm(perturbation)
print(norm)

0.28813570485770174


In [422]:
# run UCB with perturbation
ucb_with_perturb = UCBAlgorithm(k, d, mu, logged_data, perturbation)
rewards, chosen_arms = ucb_with_perturb.run(T)
print("\nNumber of pulls per arm:", ucb_with_perturb.N)
print(chosen_arms)

100%|██████████| 100/100 [00:00<00:00, 28908.29it/s]


Number of pulls per arm: [ 1.  5. 72. 22.]
[0 1 2 3 1 1 3 2 3 1 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 3 3 3 3 3 3 2 3 3 3 3 3 3 3 3 3 3 3 3 3]


In [423]:
print(len(find_perturbation.history))

96


## ABLATION 3 (QP)

In [424]:
# check all inequalities (mode=1)
epsilon = 0.5
qp = True
find_perturbation = FindPerturbation(k, d, mu, logged_data, epsilon, qp)
chosen_arms = find_perturbation.run(T, mode=1)
print("\nNumber of pulls per arm:", find_perturbation.N)
print(chosen_arms)

100%|██████████| 100/100 [02:23<00:00,  1.43s/it]


Number of pulls per arm: [ 1. 33. 33. 33.]
[0 1 2 3 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1
 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2
 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1]


In [425]:
perturbation = find_perturbation.perturbation
norm = np.linalg.norm(perturbation)
print(norm)

0.27159702498323224


In [426]:
# run UCB with perturbation
ucb_with_perturb = UCBAlgorithm(k, d, mu, logged_data, perturbation)
rewards, chosen_arms = ucb_with_perturb.run(T)
print("\nNumber of pulls per arm:", ucb_with_perturb.N)
print(chosen_arms)

100%|██████████| 100/100 [00:00<00:00, 29413.07it/s]


Number of pulls per arm: [ 1. 33. 33. 33.]
[0 1 2 3 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1
 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2
 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1]


In [427]:
print(len(find_perturbation.history))

288


## ABLATION 4 (M alternatives)

In [457]:
# check all inequalities (mode=1)
epsilon = 0.5
qp = False
alternatives = True
M = 10 # alternatives
find_perturbation = FindPerturbation(k, d, mu, logged_data, epsilon, qp, M)
chosen_arms = find_perturbation.run(T, mode=1)
print("\nNumber of pulls per arm:", find_perturbation.N)
print(chosen_arms)

100%|██████████| 100/100 [00:00<00:00, 100.54it/s]


Number of pulls per arm: [ 1. 33. 33. 33.]
[0 1 2 3 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1
 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2
 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1 2 3 1]


In [458]:
perturbation = find_perturbation.perturbation
norm = np.linalg.norm(perturbation)
print(norm)

0.4250782840413347


In [459]:
# run UCB with perturbation
ucb_with_perturb = UCBAlgorithm(k, d, mu, logged_data, perturbation)
rewards, chosen_arms = ucb_with_perturb.run(T)
print("\nNumber of pulls per arm:", ucb_with_perturb.N)
print(chosen_arms)

100%|██████████| 100/100 [00:00<00:00, 32078.81it/s]


Number of pulls per arm: [ 1.  3. 91.  5.]
[0 1 2 3 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 3 3 2 2 2 2 2 2 2 2 2 1 1 2 3 3]


In [460]:
print(len(find_perturbation.history))

30


### ETC Attack